# Latent-Based Retrieval Demo

This notebook demonstrates simple nearest-neighbor retrieval using exported latent vectors.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors


## Load Latents

In [ ]:
LATENTS_PATH = Path("../runs/gan_ae_cfpd/latents.npy")

if not LATENTS_PATH.exists():
    raise FileNotFoundError(f"Latent file not found: {LATENTS_PATH}")

latents = np.load(LATENTS_PATH)
latents_flat = latents.reshape(latents.shape[0], -1)

print("Latent shape:", latents.shape)
print("Flattened shape:", latents_flat.shape)


## Nearest Neighbor Search

In [ ]:
k = 5
query_index = 0

nn = NearestNeighbors(n_neighbors=k, metric="cosine")
nn.fit(latents_flat)

distances, indices = nn.kneighbors(latents_flat[query_index].reshape(1, -1))

print("Query index:", query_index)
print("Nearest neighbor indices:", indices[0])
print("Cosine distances:", distances[0])


## Cosine Similarity Ranking

In [ ]:
similarities = cosine_similarity(latents_flat[query_index].reshape(1, -1), latents_flat)[0]
top_k_indices = np.argsort(similarities)[::-1][:k]

print("Top-k similar indices:", top_k_indices)
print("Top-k cosine similarities:", similarities[top_k_indices])


## Optional: Display Retrieved Images

In [ ]:
# Optional: if you have image paths matching the latent order, add them here.
# Example:
# image_paths = sorted(Path("../data/celeba/img_align_celeba").glob("*.jpg"))

image_paths = []

if image_paths:
    fig, axes = plt.subplots(1, k, figsize=(15, 3))
    for ax, idx in zip(axes, top_k_indices):
        img = plt.imread(image_paths[idx])
        ax.imshow(img)
        ax.set_title(f"Index {idx}")
        ax.axis("off")
    plt.show()
else:
    print("No image paths provided. Retrieval indices are shown above.")
